In [1]:
import requests
import pandas as pd

API_KEY = "coByCYjHfG3oSYojDuPpwOwJrs9B3R38fDXhBnsj"

url = "https://api.eia.gov/v2/electricity/rto/region-sub-ba-data/data/"

params = {
    "api_key": API_KEY,
    "frequency": "hourly",
    "data[0]": "value",
    "facets[subba][]": ["PGAE", "SCE", "SDGE", "VEA"],  # contoh subregion CAISO
    "start": "2026-05-02T00",
    "sort[0][column]": "period",
    "sort[0][direction]": "desc",
    "offset": 0,
    "length": 5000
}

response = requests.get(url, params=params)
response.raise_for_status()

json_data = response.json()

df = pd.DataFrame(json_data["response"]["data"])

print(df.head())
print(df.columns)

          period subba                   subba-name parent  \
0  2026-05-06T07  PGAE     Pacific Gas and Electric   CISO   
1  2026-05-06T07   SCE   Southern California Edison   CISO   
2  2026-05-06T07  SDGE   San Diego Gas and Electric   CISO   
3  2026-05-06T07   VEA  Valley Electric Association   CISO   
4  2026-05-06T06  PGAE     Pacific Gas and Electric   CISO   

                              parent-name  value    value-units  
0  California Independent System Operator  10454  megawatthours  
1  California Independent System Operator  10398  megawatthours  
2  California Independent System Operator   2249  megawatthours  
3  California Independent System Operator     49  megawatthours  
4  California Independent System Operator  11121  megawatthours  
Index(['period', 'subba', 'subba-name', 'parent', 'parent-name', 'value',
       'value-units'],
      dtype='object')


In [2]:
df["period"] = pd.to_datetime(df["period"])
df["value"] = pd.to_numeric(df["value"], errors="coerce")

df = df.rename(columns={
    "period": "timestamp",
    "subba": "region_code",
    "subba-name": "region_name",
    "parent": "parent_region_code",
    "parent-name": "parent_region_name",
    "value": "load_mwh"
})

df = df.sort_values(["region_code", "timestamp"])

df.head()

,timestamp,region_code,region_name,parent_region_code,parent_region_name,load_mwh,value-units
412,2026-05-02 00:00:00,PGAE,Pacific Gas and Electric,CISO,California Independent System Operator,10331,megawatthours
408,2026-05-02 01:00:00,PGAE,Pacific Gas and Electric,CISO,California Independent System Operator,10813,megawatthours
404,2026-05-02 02:00:00,PGAE,Pacific Gas and Electric,CISO,California Independent System Operator,11606,megawatthours
400,2026-05-02 03:00:00,PGAE,Pacific Gas and Electric,CISO,California Independent System Operator,12201,megawatthours
396,2026-05-02 04:00:00,PGAE,Pacific Gas and Electric,CISO,California Independent System Operator,12363,megawatthours


In [3]:
df.to_csv("eia_hourly_load.csv", index=False)

In [4]:
import requests
import pandas as pd

API_KEY = "coByCYjHfG3oSYojDuPpwOwJrs9B3R38fDXhBnsj"  # ganti dengan API key asli kamu

url = "https://api.eia.gov/v2/electricity/rto/daily-region-sub-ba-data/data/"

params = {
    "api_key": API_KEY,
    "frequency": "daily",
    "data[0]": "value",
    "start": "2026-05-02",
    "sort[0][column]": "period",
    "sort[0][direction]": "desc",
    "offset": 0,
    "length": 5000
}

response = requests.get(url, params=params)

print("Status code:", response.status_code)
print("URL:", response.url)
print(response.text[:500])

json_data = response.json()

df = pd.DataFrame(json_data["response"]["data"])

print(df.shape)
print(df.columns)
df.head()

Status code: 200
URL: https://api.eia.gov/v2/electricity/rto/daily-region-sub-ba-data/data/?api_key=coByCYjHfG3oSYojDuPpwOwJrs9B3R38fDXhBnsj&frequency=daily&data%5B0%5D=value&start=2026-05-02&sort%5B0%5D%5Bcolumn%5D=period&sort%5B0%5D%5Bdirection%5D=desc&offset=0&length=5000
{"response":{"total":"1038","dateFormat":"YYYY-MM-DD","frequency":"daily","data":[{"period":"2026-05-05","subba":"PGAE","subba-name":"Pacific Gas and Electric","parent":"CISO","parent-name":"California Independent System Operator","timezone":"Arizona","value":"263367","value-units":"megawatthours"},{"period":"2026-05-05","subba":"PGAE","subba-name":"Pacific Gas and Electric","parent":"CISO","parent-name":"California Independent System Operator","timezone":"Central","value":"263340","value-units"
(1038, 8)
Index(['period', 'subba', 'subba-name', 'parent', 'parent-name', 'timezone',
       'value', 'value-units'],
      dtype='object')


,period,subba,subba-name,parent,parent-name,timezone,value,value-units
0,2026-05-05,PGAE,Pacific Gas and Electric,CISO,California Independent System Operator,Arizona,263367,megawatthours
1,2026-05-05,PGAE,Pacific Gas and Electric,CISO,California Independent System Operator,Central,263340,megawatthours
2,2026-05-05,PGAE,Pacific Gas and Electric,CISO,California Independent System Operator,Eastern,263274,megawatthours
3,2026-05-05,PGAE,Pacific Gas and Electric,CISO,California Independent System Operator,Mountain,263404,megawatthours
4,2026-05-05,PGAE,Pacific Gas and Electric,CISO,California Independent System Operator,Pacific,263367,megawatthours


In [5]:
df[df["subba"] == "AEIC"].head()

,period,subba,subba-name,parent,parent-name,timezone,value,value-units


In [6]:
df["subba"].unique()

array(['PGAE', 'SCE', 'SDGE', 'VEA', '0001', '0004', '0006', '0027',
       '0035', '8910', 'ZONA', 'ZONB', 'ZONC', 'ZOND', 'ZONE', 'ZONF',
       'ZONG', 'ZONH', 'ZONI', 'ZONJ', 'ZONK', 'ACMA', 'CYGA', 'Frep',
       'Jica', 'KAFB', 'LAC', 'PNM', 'TSGT', 'COAS', 'EAST', 'FWES',
       'NCEN', 'NRTH', 'SCEN', 'SOUT', 'WEST', 'AE', 'AEP', 'AP', 'ATSI',
       'BC', 'CE', 'DAY', 'DEOK', 'DOM', 'DPL', 'DUQ', 'EKPC', 'JC', 'ME',
       'PE', 'PEP', 'PL', 'PN', 'PS', 'RECO', 'CSWS', 'EDE', 'GRDA',
       'INDN', 'KACY', 'KCPL', 'LES', 'MPS', 'NPPD', 'OKGE', 'OPPD',
       'SECI', 'SPRM', 'SPS', 'WAUE', 'WFEC', 'WR'], dtype=object)

In [ ]:
import requests
import pandas as pd

url = "https://api.eia.gov/v2/electricity/rto/region-data/data/"

params = {
    "api_key": API_KEY,
    "frequency": "hourly",
    "data[0]": "value",
    "facets[respondent][]": ["AECI"],
    "facets[type][]": ["D", "DF", "NG", "TI"],
    # "start": "2026-05-04T00",
    "sort[0][column]": "period",
    "sort[0][direction]": "desc",
    "offset": 0,
    "length": 5000
}

# https://api.eia.gov/v2/electricity/rto/region-data/data/?frequency=hourly&data[0]=value&facets[respondent][]=AECI&facets[type][]=D&facets[type][]=DF&facets[type][]=NG&facets[type][]=TI&start=2026-05-04T00&sort[0][column]=period&sort[0][direction]=desc&offset=0&length=5000

response = requests.get(url, params=params)

print("Status code:", response.status_code)
print("URL:", response.url)
print(response.text[:500])

response.raise_for_status()

json_data = response.json()
df = pd.DataFrame(json_data["response"]["data"])

print(df.shape)
print(df.columns)
df.head()

Status code: 200
URL: https://api.eia.gov/v2/electricity/rto/region-data/data/?api_key=coByCYjHfG3oSYojDuPpwOwJrs9B3R38fDXhBnsj&frequency=hourly&data%5B0%5D=value&facets%5Brespondent%5D%5B%5D=AECI&facets%5Btype%5D%5B%5D=D&facets%5Btype%5D%5B%5D=DF&facets%5Btype%5D%5B%5D=NG&facets%5Btype%5D%5B%5D=TI&start=2026-05-04T00&sort%5B0%5D%5Bcolumn%5D=period&sort%5B0%5D%5Bdirection%5D=desc&offset=0&length=5000
{"response":{"total":"151","dateFormat":"YYYY-MM-DD\"T\"HH24","frequency":"hourly","data":[{"period":"2026-05-06T05","respondent":"AECI","respondent-name":"Associated Electric Cooperative, Inc.","type":"DF","type-name":"Day-ahead demand forecast","value":"1701","value-units":"megawatthours"},{"period":"2026-05-06T04","respondent":"AECI","respondent-name":"Associated Electric Cooperative, Inc.","type":"DF","type-name":"Day-ahead demand forecast","value":"1855","value-units":"megawatthours"},{"per
(151, 7)
Index(['period', 'respondent', 'respondent-name', 'type', 'type-name', 'value',
      

,period,respondent,respondent-name,type,type-name,value,value-units
0,2026-05-06T05,AECI,"Associated Electric Cooperative, Inc.",DF,Day-ahead demand forecast,1701,megawatthours
1,2026-05-06T04,AECI,"Associated Electric Cooperative, Inc.",DF,Day-ahead demand forecast,1855,megawatthours
2,2026-05-06T03,AECI,"Associated Electric Cooperative, Inc.",DF,Day-ahead demand forecast,2024,megawatthours
3,2026-05-06T02,AECI,"Associated Electric Cooperative, Inc.",DF,Day-ahead demand forecast,2088,megawatthours
4,2026-05-06T01,AECI,"Associated Electric Cooperative, Inc.",DF,Day-ahead demand forecast,2046,megawatthours
